In [1]:
import sys
sys.path.append(r'C:\Users\julia\OneDrive\Escritorio\Trabajo\building_ml_models_for_protein_science\src')

In [2]:
import pandas as pd
from building_models.utils.constants import (MAX_LENGTH_SEQUENCE, MIN_LENGTH_SEQUENCE,
                                             CANONICAL_RESIDUES)
from building_models.utils.utils_functions import UtilsFunctions

In [3]:
path_to_export = "../../processed_dataset/"

In [4]:
df_data = pd.read_csv(f"{path_to_export}/merged_data/processed_dataset.csv")
df_sequences_with_errors = pd.read_csv(f"{path_to_export}/merged_data/sequences_with_erros.csv")

- Adding lengths to pending data

In [5]:
df_sequences_with_errors["length"] = df_sequences_with_errors["sequence"].str.len()

- Checking canonical residues

In [6]:
df_data["is_canon"] = df_data["sequence"].apply(UtilsFunctions.checking_canonical_residues, args=(False))
df_sequences_with_errors["is_canon"] = df_sequences_with_errors["sequence"].apply(UtilsFunctions.checking_canonical_residues, args=(False))

In [7]:
df_data["is_canon"].value_counts()

is_canon
True     7774
False      53
Name: count, dtype: int64

In [8]:
df_sequences_with_errors["is_canon"].value_counts()

is_canon
True    43
Name: count, dtype: int64

- Checking lengths

In [9]:
MIN_LENGTH_SEQUENCE, MAX_LENGTH_SEQUENCE

(2, 1024)

In [10]:
MIN_LENGTH_SEQUENCE = 70

In [11]:
df_data["is_in_length"] = df_data["length"].between(MIN_LENGTH_SEQUENCE, MAX_LENGTH_SEQUENCE)
df_data["is_in_length"].value_counts()

is_in_length
True     4221
False    3606
Name: count, dtype: int64

In [12]:
df_sequences_with_errors["is_in_length"] = df_sequences_with_errors["length"].between(MIN_LENGTH_SEQUENCE, MAX_LENGTH_SEQUENCE)
df_sequences_with_errors["is_in_length"].value_counts()

is_in_length
True     40
False     3
Name: count, dtype: int64

- Making filters

In [13]:
df_data_filtered = df_data[(df_data["is_canon"]) & (df_data["is_in_length"])]
df_sequences_with_errors_filtered = df_sequences_with_errors[(df_sequences_with_errors["is_canon"]) & (df_sequences_with_errors["is_in_length"])]

df_data_filtered.shape[0], df_sequences_with_errors_filtered.shape[0]

(4193, 40)

- Creating folder and exports

In [14]:
dict_metadata = {
    "total_data":{
        "processed_sequences" : df_data.shape[0],
        "sequences_with_errors" : df_sequences_with_errors.shape[0],
    },
    "canonical_sequences":{
        "used_vocab" : CANONICAL_RESIDUES,
        "processed_sequences":{
            "is_canon" : df_data[df_data["is_canon"]].shape[0],
            "is_not_canon" : df_data[df_data["is_canon"]==False].shape[0],
        },
        "sequences_with_error":{
            "is_canon" : df_sequences_with_errors[df_sequences_with_errors["is_canon"]].shape[0],
            "is_not_canon" : df_sequences_with_errors[df_sequences_with_errors["is_canon"]==False].shape[0],
        }
    },
    "length_evaluation":{
        "defined_length":{
            "min_value" : MIN_LENGTH_SEQUENCE,
            "max_value" : MAX_LENGTH_SEQUENCE,
        },
        "processed_sequences":{
            "is_in_length" : df_data[df_data["is_in_length"]].shape[0],
            "is_not_in_length" : df_data[df_data["is_in_length"]==False].shape[0],
        },
        "sequences_with_error" : {
            "is_in_length" : df_sequences_with_errors[df_sequences_with_errors["is_in_length"]].shape[0],
            "is_not_in_length" : df_sequences_with_errors[df_sequences_with_errors["is_in_length"] == False].shape[0],
        }
    },
    "final_evaluation":{
        "processed_sequences" : df_data_filtered.shape[0],
        "positive_examples" : df_data_filtered[df_data_filtered["label"] == 1].shape[0],
        "negative_examples" : df_data_filtered[df_data_filtered["label"] == 0].shape[0],
        "sequences_with_errors" : df_sequences_with_errors_filtered.shape[0],
    }
}

dict_metadata

{'total_data': {'processed_sequences': 7827, 'sequences_with_errors': 43},
 'canonical_sequences': {'used_vocab': ['A',
   'C',
   'D',
   'E',
   'F',
   'G',
   'H',
   'I',
   'K',
   'L',
   'M',
   'N',
   'P',
   'Q',
   'R',
   'S',
   'T',
   'V',
   'W',
   'Y'],
  'processed_sequences': {'is_canon': 7774, 'is_not_canon': 53},
  'sequences_with_error': {'is_canon': 43, 'is_not_canon': 0}},
 'length_evaluation': {'defined_length': {'min_value': 70, 'max_value': 1024},
  'processed_sequences': {'is_in_length': 4221, 'is_not_in_length': 3606},
  'sequences_with_error': {'is_in_length': 40, 'is_not_in_length': 3}},
 'final_evaluation': {'processed_sequences': 4193,
  'positive_examples': 1010,
  'negative_examples': 3183,
  'sequences_with_errors': 40}}

- Exporting data

In [15]:
UtilsFunctions.make_directory(f"{path_to_export}processed_data")

In [16]:
UtilsFunctions.export_json(f"{path_to_export}processed_data/metadata_process.json", dict_metadata)

In [17]:
df_data_filtered.to_csv(f"{path_to_export}processed_data/processed_dataset.csv", index=False)
df_sequences_with_errors_filtered.to_csv(f"{path_to_export}processed_data/sequences_with_errors.csv", index=False)